**Лабораторная работа №9**

In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve

1. Загрузите файл classification.csv. В нем записаны истинные классы объектов выборки (колонка true) и ответы некоторого классификатора (колонка predicted).

In [5]:
df_class = pd.read_csv('classification.csv')

2. Заполните таблицу ошибок классификации. Для этого подсчитайте величины TP, FP, FN и TN согласно их определениям. Ответ в данном вопросе — четыре числа через пробел.

In [7]:
TP = ((df_class['true'] == 1) & (df_class['pred'] == 1)).sum()
FP = ((df_class['true'] == 0) & (df_class['pred'] == 1)).sum()
FN = ((df_class['true'] == 1) & (df_class['pred'] == 0)).sum()
TN = ((df_class['true'] == 0) & (df_class['pred'] == 0)).sum()

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")
print(f"\n{TP} {FP} {FN} {TN}")

TP = 43
FP = 34
FN = 59
TN = 64

43 34 59 64


3. Посчитайте основные метрики качества классификатора:
Accuracy, Precision, Recall, F-мера

In [10]:
y_true = df_class['true']
y_pred = df_class['pred']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy = {accuracy:.4f}")
print(f"Precision = {precision:.4f}")
print(f"Recall = {recall:.4f}")
print(f"F1 = {f1:.4f}")

Accuracy = 0.5350
Precision = 0.5584
Recall = 0.4216
F1 = 0.4804


4. Имеется четыре обученных классификатора. В файле scores.csv записаны истинные классы и значения степени принадлежности положительному классу для каждого классификатора на некоторой
выборке:
• для логистической регрессии — вероятность положительного класса (колонка score_logreg),
• для SVM — отступ от разделяющей поверхности (колонка score_svm),
• для метрического алгоритма — взвешенная сумма классов соседей (колонка score_knn),
• для решающего дерева — доля положительных объектов в листе (колонка score_tree)

In [15]:
df_scores = pd.read_csv('scores.csv')
y_true_scores = df_scores['true']

models = ['score_logreg', 'score_svm', 'score_knn', 'score_tree']
auc_scores = {}

for model in models:
    y_score = df_scores[model]
    auc = roc_auc_score(y_true_scores, y_score)
    auc_scores[model] = auc
    print(f"AUC-ROC для {model:15s} = {auc:.4f}")

AUC-ROC для score_logreg    = 0.7192
AUC-ROC для score_svm       = 0.7087
AUC-ROC для score_knn       = 0.6352
AUC-ROC для score_tree      = 0.6919


5. Посчитайте площадь под ROC-кривой для каждого классификатора. Какой классификатор имеет наибольшее значение метрики AUC-ROC (укажите название столбца с ответами этого классификатора)?

In [16]:
best_auc_model = max(auc_scores, key=auc_scores.get)
print(f"\nНаибольшее значение AUC-ROC у классификатора: {best_auc_model}")


Наибольшее значение AUC-ROC у классификатора: score_logreg


6. Какой классификатор достигает наибольшей точности (Precision) при полноте (Recall) не менее 70% (укажите название столбца с ответами этого классификатора)? Какое значение точности при этом
получается?

In [22]:
best_precision_at_recall70 = {}
best_overall_precision = -1
best_overall_model = None

for model in models:
    y_score = df_scores[model]
    precision_vals, recall_vals, thresholds = precision_recall_curve(y_true_scores, y_score)

    mask = recall_vals >= 0.7
    if np.any(mask):
        max_precision = np.max(precision_vals[mask])
        best_precision_at_recall70[model] = max_precision
        print(f"Модель {model:15s}: максимальная Precision при Recall >= 0.7 = {max_precision:.4f}")
        if max_precision > best_overall_precision:
            best_overall_precision = max_precision
            best_overall_model = model
    else:
        best_precision_at_recall70[model] = None
        print(f"Модель {model:15s}: нет точек с Recall >= 0.7")

print(f"\nЛучшая точность при Recall >= 0.7 достигается у модели: {best_overall_model}")
print(f"Значение точности = {best_overall_precision:.4f}")

Модель score_logreg   : максимальная Precision при Recall >= 0.7 = 0.6303
Модель score_svm      : максимальная Precision при Recall >= 0.7 = 0.6228
Модель score_knn      : максимальная Precision при Recall >= 0.7 = 0.6066
Модель score_tree     : максимальная Precision при Recall >= 0.7 = 0.6518

Лучшая точность при Recall >= 0.7 достигается у модели: score_tree
Значение точности = 0.6518
